# Worksheet 2.2: Hot and Cold LLMs

Today we'll explore a key setting that controls how AI generates text, and actually *see* the probabilities that AIs use when writing. This is the look behind the scenes that the video before class started.

## How your screen works

Same as last class, with one new rule for choosing the first Driver.

- Sit **side by side**, so you can all see the screen.
- The **Driver** types in the notebook.
- The **Navigator** reads each instruction and exercise **out loud**, then tells the Driver what to do. You can read it word for word, or say it in your own words, in English or Vietnamese.
- If there are three of you, the third person is the **Checker**, who makes sure your answers really answer what each exercise asks.
- You will **swap Drivers once**, about halfway through. This worksheet shows you where.
- **Who drives first?** If you did not drive last class, you drive first today.

## Getting started

Just like last class:

1. **File > Save a copy in Drive**, then close the old tab (the one whose name does *not* start with "Copy of").
2. Click **Share**, and add everyone in your screen and the instructor as **Editors**.
3. Someone other than the Driver opens the link on another device to check it works, then closes that tab.

**IMPORTANT:** Only one person should have the notebook open at a time.

## Load the AI

**Run the cell below right away.** It loads a small AI model into this notebook, which takes about a minute, so it can get going while you read the next part.

**NOTE:** This AI is much smaller than ChatGPT, so it acts differently.

In [ ]:
#@title Run this cell first (takes about a minute)

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

print("Loading the AI (about 700 MB, one time only)... ", end="")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float()   # full precision: faster on CPU, and charts need it
model.eval()
VOCAB_SIZE = len(tokenizer)
print("Done!")

def _show(token_id):
    """Show a token so that spaces are visible."""
    return repr(tokenizer.decode(token_id))

def _next_logits(text):
    ids = tokenizer(text, return_tensors="pt").input_ids
    with torch.no_grad():
        return model(ids).logits[0, -1, :]

def _is_word_start(token_id):
    """True if this piece begins a new word (a space, a newline, or punctuation)."""
    s = tokenizer.decode(token_id)
    return s == "" or not s[0].isalnum()

def _whole_word(text, token_id, max_extra=4):
    """The AI sometimes predicts a piece of a word, like ' H'. Finish the word with the
    AI's most likely next pieces, so the tables show whole words, like ' Hanoi'."""
    word = tokenizer.decode(token_id)
    if not word.strip() or not word.strip()[0].isalnum():
        return word                      # punctuation stays as it is
    ids = tokenizer(text, return_tensors="pt").input_ids
    seq = torch.cat([ids, torch.tensor([[token_id]])], dim=1)
    for _ in range(max_extra):
        with torch.no_grad():
            nxt = model(seq).logits[0, -1].argmax().item()
        if _is_word_start(nxt):
            break
        word += tokenizer.decode(nxt)
        seq = torch.cat([seq, torch.tensor([[nxt]])], dim=1)
    return word

# ============================================================
# FUNCTION 0: Ask the AI a question
# ============================================================
def ask_ai(prompt, temperature=1.0, max_words=50):
    """Ask the AI a question and print its answer."""
    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    settings = dict(max_new_tokens=max_words, pad_token_id=tokenizer.eos_token_id)
    if temperature <= 0.01:
        settings["do_sample"] = False
    else:
        settings.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    with torch.no_grad():
        out = model.generate(**enc, **settings)
    answer = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(answer.strip())

# ============================================================
# FUNCTION 1: Show next word probabilities
# ============================================================
def show_next_word_probabilities(text, top_k=10):
    """Show what the AI thinks are the most likely next words."""
    probs = F.softmax(_next_logits(text), dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)

    print(f'Prompt: "{text}"')
    print(f"\nTop {top_k} predictions for the next word:\n")
    for i in range(top_k):
        prob = top_probs[i].item() * 100
        bar = "*" * int(prob / 2) + "." * (50 - int(prob / 2))
        word = repr(_whole_word(text, top_indices[i].item()))
        print(f"  {word:15} {bar} {prob:5.1f}%")

# ============================================================
# FUNCTION 2: Show temperature effect with "other" category
# ============================================================
def show_temperature_effect(text, temperatures=[0.5, 1.0, 2.0], top_k=5):
    """Show how temperature changes the probability distribution."""
    logits = _next_logits(text)
    # Temperature never changes the ORDER of the words, only their probabilities,
    # so the top words can be finished once and reused for every temperature.
    top_ids = torch.topk(logits, top_k).indices.tolist()
    words = {tid: repr(_whole_word(text, tid)) for tid in top_ids}
    print(f'Prompt: "{text}"\n')
    for temp in temperatures:
        probs = F.softmax(logits / temp, dim=-1)
        top_probs, top_indices = torch.topk(probs, top_k)
        other_prob = 100 - top_probs.sum().item() * 100

        print("=" * 55)
        print(f"Temperature {temp}")
        print("=" * 55)
        for i in range(top_k):
            prob = top_probs[i].item() * 100
            bar = "*" * int(prob / 2) + "." * (50 - int(prob / 2))
            print(f"  {words[top_indices[i].item()]:12} {bar} {prob:5.1f}%")
        other_bar = "*" * int(other_prob / 2) + "." * (50 - int(other_prob / 2))
        print(f"  {'[other]':12} {other_bar} {other_prob:5.1f}%  <- {VOCAB_SIZE - top_k:,} other words")
        print()

# ============================================================
# FUNCTION 3: Generate step by step
# ============================================================
def generate_step_by_step(prompt, num_words=8, temperature=1.0):
    """Generate text and show what was chosen at each step."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    print(f'Starting prompt: "{prompt}"')
    print(f"Temperature: {temperature}\n")
    print("Step-by-step generation:")
    print("=" * 60)

    for step in range(num_words):
        with torch.no_grad():
            logits = model(input_ids).logits[0, -1, :]
        probs = F.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        chosen = next_token.item()
        top_probs, top_indices = torch.topk(probs, 3)

        print(f"\nStep {step + 1}: Top 3 options were:")
        for i in range(3):
            tid = top_indices[i].item()
            mark = " <- CHOSEN" if tid == chosen else ""
            print(f"  {_show(tid)}: {top_probs[i].item() * 100:.1f}%{mark}")
        if chosen not in top_indices.tolist():
            print("     ...")
            print("  (many other words...)")
            print("     ...")
            print(f"  {_show(chosen)}: {probs[chosen].item() * 100:.2f}% <- CHOSEN (surprise pick!)")

        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    print("\n" + "=" * 60)
    print(f'Final result: "{tokenizer.decode(input_ids[0], skip_special_tokens=True)}"')

print("\n" + "=" * 50)
print("The AI is ready! Functions available:")
print("  - ask_ai(prompt, temperature=1.0)")
print("  - show_next_word_probabilities(text)")
print("  - show_temperature_effect(text)")
print("  - generate_step_by_step(prompt, temperature=1.0)")
print("=" * 50)

# Using AI on this worksheet

Today the AI is the thing we are studying, so you will be running one right here in this notebook. Please don't use any other AI tools, like ChatGPT, on this worksheet. When a question asks what you noticed, predicted, or think, the answer should come from your screen.

**Exercise 0. Double-click to edit the text cell below, and write "We agree" at the end:**

We, the people in this screen, understand that we will only use AI where this worksheet asks us to, and that our answers about what we noticed and what we think will be our own. ______________

# Part A: What Does Temperature Do? (Discovery)

In the opening activity, we built sentences by voting on the next word. But how does the AI decide which words to offer, and how does it choose between them?

Let's experiment with a setting called `temperature` and see if we can figure out what it does, using the AI you loaded at the start.

### Write your prompts

Decide on your two prompts together as a screen.

In [ ]:
# Prompt A: Something with ONE correct answer
# Examples: "What is the capital of Japan? Answer in one sentence."
my_factual_prompt = "___"

# Prompt B: Something creative
# Example: "Write 2-3 sentences about a robot who learns to dance."
my_creative_prompt = "___"

### Experiment Round 1

Run each cell **3 times**. Pay attention to the outputs. Each answer takes a few seconds.

In [ ]:
# Run this cell 3 times
ask_ai(my_factual_prompt, temperature=0.01)

In [ ]:
# Run this cell 3 times
ask_ai(my_creative_prompt, temperature=0.01)

Discuss with your screen: what do you notice? Nothing to write down here.

### Experiment Round 2

Same prompts. Run each cell **3 times** again.

In [ ]:
# Run this cell 3 times
ask_ai(my_factual_prompt, temperature=1.2)

In [ ]:
# Run this cell 3 times
ask_ai(my_creative_prompt, temperature=1.2)

**Exercise A.1:** Look again at your answers from Round 1 and Round 2. Write **one word** that describes the Round 1 answers, and **one word** that describes the Round 2 answers.

- Round 1 (temperature 0.01):
- Round 2 (temperature 1.2):

Then **write your two words on the board** for the class.

*Your answer here.*

# Part B: Looking Inside the AI's Brain

You have seen what changes when the temperature goes up. Now let's look inside the AI to see what is actually happening.

We can do this because the AI we're using, SmolLM2, is fully **open**: anyone can download it and look at the numbers it uses. It was made in 2024 by Hugging Face, the same company from last class's story.

### What does the AI think comes next?

When you give the AI some text, it calculates a **probability for every possible next word** (all 49,152 of them!).

Let's see what it thinks should come after "The weather today is":

In [ ]:
show_next_word_probabilities("The weather today is")

**NOTE:** The quotation marks around each word show you where the spaces are. Here's a secret: the AI really works with *pieces* of words, called **tokens**. When its guess is only the start of a word, like `' H'`, this table finishes the word the way the AI most likely would, like `' Hanoi'`. You'll see the pieces themselves in Part D.

**Exercise B.1:** Here is a line of code. Don't run anything yet. What do you think will happen when we run it?

```python
show_next_word_probabilities(Once upon a time)
```

*Your answer here.*

Now run it:

In [ ]:
show_next_word_probabilities(Once upon a time)

**Exercise B.2:** What happened? Why do you think Python did that? (Think back to Worksheet 2.1.)

*Your answer here.*

# Part C: How Temperature Changes the Probabilities

Now let's see how temperature changes the probabilities.

First, **your screen picks a sentence** for the AI to continue. Replace the blank with the
*beginning* of a sentence, not a question, and make it something you actually care about: food, a
place, a band, a game, your home town. Then run the cell.

**NOTE:** Finish on a word, with nothing after it. No period, no comma, no space. This AI is
guessing the *next* word, so `"My favorite food in Hanoi is"` works well, while
`"My favorite food in Hanoi is."` gives it nothing to continue.

In [ ]:
my_sentence = "___"

In [ ]:
show_temperature_effect(my_sentence, temperatures=[0.5, 1.0, 2.0])

**Exercise C.1:** Look at the percentages AND the "[other]" category. What happens as temperature increases?

- The TOP word's probability:
- The "[other]" category (all the remaining words):

*Your answer here.*

**Exercise C.2:** Here is another line of code. Don't run it yet. Which words do you think will have the highest probability?

```python
show_next_word_probabilities("The capital of Vietnam is")
```

*Your answer here.*

Now run it:

In [ ]:
show_next_word_probabilities("The capital of Vietnam is")

**Exercise C.3:** What did the AI predict? Was it what you expected?

*Your answer here.*

## 🔄 Time to swap Drivers

The second Driver takes over now.

- If the new Driver wants to use their own laptop, they open the notebook link, and the old Driver **closes their tab**.
- Or, if you're both happy to, just pass the laptop over.

Either way, only one person types at a time.

**NOTE:** If the new Driver opens the notebook on their own laptop, two cells have to be run again on that laptop before Part D will work: the **Load the AI** cell near the top, and the `my_sentence` cell in Part C. Nothing you wrote is lost. That laptop just hasn't run them yet.

# Part D: How Does Sampling Work?

The AI doesn't just pick the highest probability word. It **samples** from the probabilities, a bit like our opening activity.

Let's watch the AI generate text step by step:

In [ ]:
generate_step_by_step(my_sentence, temperature=1.0)

**Exercise D.1:** Run the cell above a few times. Does the AI always choose the highest probability word? Did you see any "surprise picks"?

*Your answer here.*

Now try with different temperatures:

In [ ]:
# Low temperature
generate_step_by_step(my_sentence, temperature=0.3)

In [ ]:
# High temperature
generate_step_by_step(my_sentence, temperature=2.0)

**Exercise D.2:** Compare the three outputs (temperature 0.3, 1.0, and 2.0). How many "surprise picks" (words outside the top 3) did you see at each temperature?

*Your answer here.*

# Part E: The Big Picture

**Exercise E.1:** How does this connect to the "Shoggoth" idea from Class 1.2? What does temperature reveal about the nature of LLMs?

*Your answer here.*

# Extra: Explore the Distribution (if you finish early)

This part is **not required**. If your screen has time left, try it!

Let's see exactly how temperature transforms the probabilities, with a slider you can drag.

In [ ]:
#@title Run this cell to set up the interactive visualization

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def create_temperature_visualization(prompt):
    """Create an interactive visualization of temperature's effect on the distribution."""
    logits = _next_logits(prompt)
    words = [repr(_whole_word(prompt, tid)) for tid in torch.topk(logits, 5).indices.tolist()]
    output = widgets.Output()
    temp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                      description='Temperature:', continuous_update=False,
                                      readout_format='.1f', style={'description_width': '100px'},
                                      layout=widgets.Layout(width='500px'))

    def draw(temperature):
        probs = F.softmax(logits / temperature, dim=-1)
        top_probs, top_indices = torch.topk(probs, 100)
        sorted_probs, _ = torch.sort(probs, descending=True)
        words_for_90 = (torch.cumsum(sorted_probs, dim=0) < 0.9).sum().item() + 1

        fig = plt.figure(figsize=(12, 5))
        ax1 = fig.add_axes([0.05, 0.15, 0.55, 0.75])
        ax1.bar(range(1, 101), top_probs.numpy() * 100, color='steelblue', width=0.8)
        ax1.set_xlabel('Word rank')
        ax1.set_ylabel('Probability (%)')
        ax1.set_title(f'Top 100 next-word probabilities (temperature = {temperature:.1f})')
        ax1.set_xlim(0, 101)

        ax2 = fig.add_axes([0.68, 0.15, 0.30, 0.75])
        ax2.axis('off')
        lines = ["NUMBERS", "",
                 f"Top word probability: {top_probs[0].item() * 100:.1f}%", "",
                 f"Top 5 combined: {top_probs[:5].sum().item() * 100:.1f}%", "",
                 f"Words needed for 90%: {words_for_90:,}", "", "", "TOP 5 WORDS"]
        for i in range(5):
            lines.append(f"{i + 1}. {words[i]}: {top_probs[i].item() * 100:.1f}%")
        ax2.text(0.0, 0.95, "\n".join(lines), transform=ax2.transAxes,
                 fontsize=11, verticalalignment='top', fontfamily='monospace')
        plt.suptitle(f'Prompt: "{prompt}"', fontsize=12, y=0.98)
        return fig

    def update(change):
        with output:
            clear_output(wait=True)
            draw(change['new'])
            plt.show()

    temp_slider.observe(update, names='value')
    display(temp_slider)
    display(output)
    update({'new': 1.0})
    return draw

print("Interactive visualization ready!")

In [ ]:
#@title Enter a prompt to explore, then press "play"
visualization_prompt = "I did great on the Mini-Test because I drank a lot of" #@param {type:"string"}

create_temperature_visualization(visualization_prompt)

**Extra Exercise 1:** Keep the temperature at 1.0. Can you find a prompt where the probabilities are already "spread out" (top word under 20%)? Can you find one where they are very "focused" (top word over 50%)?

*Your answer here.*

**Extra Exercise 2:** What makes some prompts more spread out than others, even at the same temperature?

*Your answer here.*

# Important: Submission Instructions

1. Check to make sure you've completed all the exercises (the Extra ones are optional).
2. Check that the notebook is shared as an **Editor** with everyone in your screen and the instructor.
3. **Each of you** submits on Canvas: paste the notebook link into the **Website URL** box, and in the **Comments** box write whether you drove today (**Drove**, **Didn't drive**, or **N/A**). Everyone in your screen submits the same link.

**Acknowledgments:** This notebook contains materials created by Ethan C. Brown, collaborating with Claude (see [conversation 1](https://claude.ai/share/0a0838b1-d283-489c-a1b1-7d975b6a1b53), [conversation 2](https://claude.ai/share/4a57eac2-d618-4942-aaff-6964fe09c00a)), and heavily inspired by Tufino, E. (2025). Exploring large language models (LLMs) through interactive Python activities. *Physics Education, 60*(5), 055003. https://doi.org/10.1088/1361-6552/adea28

Current version created by Ethan C. Brown in collaboration with Claude Code.